In [ ]:
## import libs
import pandas as pd
import requests
from bs4 import BeautifulSoup
import maritalk
import re
import warnings
from io import StringIO

import os
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

In [ ]:
## carregar variaveis ​​do arquivo .env (sua chave de API deve estar aqui)
load_dotenv()

## buscando a chave da API
API_KEY = os.getenv("MARITALK_API_KEY")

if not API_KEY:
    raise ValueError("MARITALK_API_KEY nao encontrada. Verifique o arquivo .env")

# instanciando o modelo
model = maritalk.MariTalk(
    key=API_KEY,
    model="sabia-4"
)

print(f"Modelo carregado: {model.model}")

In [ ]:
headers = {"User-Agent": "Mozilla/5.0"}

while True:
    cidade = input("Digite o nome da cidade: ").strip()
    url = "https://pt.wikipedia.org/wiki/" + cidade.replace(" ", "_")

    resp = requests.get(url, headers=headers, timeout=20)
    html = resp.text

    try:
        tabelas = pd.read_html(StringIO(html), decimal=",", thousands=".")
    except ValueError:
        tabelas = []

    print(f"{len(tabelas)} tabelas encontradas para {cidade}")

    if len(tabelas) > 1:   # mantendo sua regra
        print("✅ Cidade verificada! Vamos em frente!")
        break
    else:
        print("❌ Cidade nao localizada! Por favor, tente outra cidade:")


In [89]:
def encontrar_tabela_clima(tables_found):
    """Encontra a tabela que contém dados de temperatura"""
    palavras_chave = [
        'Temperatura máxima', 
        'Temperatura média', 
        'Temperatura mínima',
        'Temperatura'
    ]
    
    for i, df in enumerate(tables_found):
        # pega todo o conteúdo da tabela como texto
        texto_tabela = df.astype(str).values.flatten()
        
        # verifica se alguma palavra-chave aparece
        for palavra in palavras_chave:
            if df.astype(str).apply(lambda col: col.str.contains(palavra, na=False, case=False)).any().any():
                return i
    
    return None  # se nao achar nenhuma

In [ ]:
# chamando a funcao acima
idx_clima = encontrar_tabela_clima(tabelas)
print(f"Tabela de clima encontrada no índice: {idx_clima}")

if idx_clima is not None:
    clima = tabelas[idx_clima]
    print("✅ Tabela carregada com sucesso!")
    # display(clima.head())
else:
    print("❌ Tabela de clima não encontrada")
    clima = 'Tabela de clima não encontrada'

In [91]:
cols = []
for i in range(len(clima.columns)):
  cols.append(clima.columns[i][1])
cols[0] = 'Medidas'

In [92]:
# processa a tabela somente se for encontrada tabela referente o clima
if 'idx_clima' in locals() and idx_clima is not None:
    clima = tabelas[idx_clima].iloc[:-1].copy()
    clima.columns = cols
    clima.index = clima['Medidas']
    clima.drop('Medidas', axis=1, inplace=True)
    if 'Ano' in clima.columns:
        clima.drop('Ano', axis=1, inplace=True)
else:
    clima = None

In [ ]:
soup = BeautifulSoup(html, "html.parser")

paragrafos = []

for p in soup.find_all("p"):
    txt = p.get_text().strip()
    if len(txt) > 80:
        paragrafos.append(txt)

print("Quantidade de parágrafos:", len(paragrafos))
print(paragrafos[0])


In [95]:
df_texto = pd.DataFrame({
    "texto": paragrafos
})

texto_unico = ' '.join(df_texto.astype(str).values.flatten())

In [ ]:
prompt_resumo_cidade = f"""
Estou estudando esta cidade para uma recomendacao de visita, observe o texto abaixo e faca um resumo de modo a passar as principais informacoes para um guia turistico.
Texto:
{texto_unico}
"""

response = model.generate(prompt_resumo_cidade,
                          max_tokens=1500,
                          num_tokens_per_message=550,
                          temperature=0.2, #0.0
                          top_p=0.99, #0.99
                          do_sample=False,
                          stream=False,
                          return_async_generator=False
                          )
resposta = response

resumo = response['answer'] if response else "Erro"
print(resumo)

In [ ]:
def adicionar_meses(df):
    meses = df.columns.tolist()  # ['Jan', 'Fev', 'Mar'...]
    
    novo_df = df.copy()
    
    # Pra CADA linha, concatena "Mês - valor"
    for idx in df.index:
        for col in meses:
            valor = df.at[idx, col]
            novo_df.at[idx, col] = f"{col} - {valor}"
    
    return novo_df

# USO
clima_com_meses = adicionar_meses(clima)
clima_com_meses

In [ ]:
def df_para_texto_narrativo(df):
    """Converte DataFrame pra texto seguro (números + strings)"""
    texto = f"Dados climáticos ({len(df)} medidas):\n\n"
    
    for idx, row in df.iterrows():
        medida = str(idx)  # sempre string segura
        
        # Trata cada valor: número → float formatado, string → texto puro
        valores = []
        for v in row.values:
            try:
                valores.append(f"{float(v):.1f}")
            except (ValueError, TypeError):
                valores.append(str(v))  # mantém string original
        
        texto += f"- {medida}: {', '.join(valores)}\n"
    
    return texto.strip()

# USO
prompt_texto = df_para_texto_narrativo(clima_com_meses)
print(prompt_texto)

In [ ]:
prompt_resumo_clima = f"""
Referencia climatica da cidade:

{prompt_texto}

Explique em linguagem simples como é o clima ao longo do ano.
Máximo 5 linhas.

e faca um resume em ate 10 palavras das caracteristicas de cada mes
"""

response = model.generate(prompt_resumo_clima,
                          max_tokens=1500,
                          num_tokens_per_message=550,
                          temperature=0.0, #0.0
                          top_p=0.99, #0.99
                          do_sample=False,
                          stream=False,
                          return_async_generator=False
                          )
resposta = response

clima_resumo = response['answer'] if response else "Erro"
print(clima_resumo)

In [100]:
cidade_perfeita = input("""

🌴 Descreva qual seria o estilo de viagem ideal para voce! (ex: "praia calma, sol, família, médio")

• Praia/montanha? 🏖️/⛰️
• Clima? ☀️/❄️  
• Vibe? 🎉/😌
• Com quem? 👨‍👩‍👧‍👦/💑
• Orçamento? 💰
""").strip()

In [101]:
prompt_sugestao = f"""
Dados climáticos:
{clima_resumo}

Descrição da cidade:
{resumo}

Perfil do cliente:
{cidade_perfeita}

Com base nas informacoes acima crie uma pontuacao de 0 a 10 para este perfil de cliente com base nos dados climaticos e na descricao da cidade.
Posterior em poucas palavras resuma o que te fez atribuir esta pontuacao.

De acordo com as informacoes dele tambem faca sugestoes de melhores periodos de viagem conforme a preferencia dele e os dados climaticos da cidade

Voce precisa trazer embasamento na sua nota por que a nota e baixa, porque a nota nao e maxima etc

Obrigatorio: Responda como se estivesse falando direto para o cliente final

Responda em JSON válido com:
{{
 "pontuacao": número,
 "resumo": texto
}}
"""

response = model.generate(prompt_sugestao,
                          max_tokens=1500,
                          num_tokens_per_message=550,
                          temperature=0.0, #0.0
                          top_p=0.99, #0.99
                          do_sample=False,
                          stream=False,
                          return_async_generator=False
                          )
resposta = response

In [102]:
def extrair_json_llm(resposta_api):
    import json, re

    texto = resposta_api["answer"] if isinstance(resposta_api, dict) and "answer" in resposta_api else str(resposta_api)

    # 1) tenta pegar bloco ```json ... ```
    m = re.search(r"```json\s*(\{.*?\})\s*```", texto, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return json.loads(m.group(1))

    # 2) fallback: pega o primeiro {...} completo no texto
    m = re.search(r"(\{.*\})", texto, flags=re.DOTALL)
    if m:
        candidato = m.group(1)

        # corta qualquer coisa depois do último "}" (se vier texto colado)
        last = candidato.rfind("}")
        candidato = candidato[: last + 1]

        return json.loads(candidato)

    raise ValueError("Não achei um JSON válido na resposta do modelo.")


In [103]:
from IPython.display import display, HTML

def formatar_resposta_llm(dados, cidade):
    pontuacao = dados.get("pontuacao", "N/A")
    resumo = dados.get("resumo", "")

    html = f"""
    <div style="
        max-width: 850px;
        margin: 30px auto;
        padding: 30px;
        background: linear-gradient(135deg, #1e3c72, #2a5298);
        border-radius: 20px;
        color: white;
        box-shadow: 0 20px 40px rgba(0,0,0,0.25);
        font-family: Arial;
    ">
        <h2>📍 Avaliação da cidade: {cidade}</h2>
        <div style="font-size:1.4em; font-weight:bold; color:#FFD700;">
            ⭐ Pontuação: {pontuacao}/10
        </div>
        <div style="
            background: rgba(255,255,255,0.12);
            padding: 20px;
            border-radius: 12px;
            margin-top: 15px;
            line-height: 1.6;
        ">
            <b>✨ Resumo:</b><br><br>
            {resumo}
        </div>
    </div>
    """
    display(HTML(html))

In [104]:
dados = extrair_json_llm(response)
formatar_resposta_llm(dados, cidade)